## Description

The goal of this notebook is to merge all EEG features, diagnosis, and demog information into 1 CSV file.

# Imports

In [1]:
# Imports
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import FileLink
import seaborn as sns

### Contants

In [2]:
isRio = False # Set to True if running Rio analysis

### Paths

In [3]:
# Directory
root_dir = "/Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/"
data_dir = os.path.join(root_dir, 'Data/')

# Original Data CSVs from REDCAP
original_demog_genetics_data = os.path.join(root_dir,'Data/Genetics/Input/Q1K report EEG_NDD_génétique.csv')
original_dia_cogn_data = os.path.join(root_dir, 'Data/Diagnosis + Cogn Tests/Q1K-Dia_cogn.csv')
original_eeg_rs_data = os.path.join(root_dir, 'Data/EEG/Q1K_concatenated_features_RS.csv') # Preprosessed EEG data with HAPPE

# Input Files for CNV prediction
genetics_only_data = os.path.join(root_dir, 'Data/Genetics/Input/CNV-Analysis.csv')
hg38_input_data = os.path.join(root_dir,'Data/Genetics/Input/CNV-Analysis-Hg38.tsv')
hg19_input_data = os.path.join(root_dir,'Data/Genetics/Input/CNV-Analysis-Hg19.tsv')
hg18_input_data = os.path.join(root_dir,'Data/Genetics/Input/CNV-Analysis-Hg18.tsv')

# Calculated CNVs files
cnv_prediction_hg19_data = os.path.join(root_dir, 'Data/Genetics/Input/cnvprediction-hg19-output.csv')
cnv_prediction_hg38_data = os.path.join(root_dir, 'Data/Genetics/Input/cnvprediction-hg38-output.csv')

# Index to Pariticpant Code Map
id_map = os.path.join(root_dir, 'Data/Genetics/Input/sample-id-map.csv')

# Output Files
preprocessed_data_path = os.path.join(root_dir, 'Data/Final/GENIAL-DB-preprocessed-RS.csv')

### Check Rio VS RS count

In [4]:
# Read the CSV file
features_df = pd.read_csv(os.path.join(root_dir, 'Data/EEG/Q1K_concatenated_features_RS.csv')).rename(columns=lambda x: x.strip())


# Count occurrences of RS and RSRIO in the IDs
rs_count = features_df['ID'].str.contains('_RS_').sum()
rsrio_count = features_df['ID'].str.contains('_RSRIO_|_RSRio_').sum()

print(f"Number of RS recordings: {rs_count}")
print(f"Number of RSRIO recordings: {rsrio_count}")
print(f"Total recordings: {len(features_df)}")

Number of RS recordings: 95
Number of RSRIO recordings: 89
Total recordings: 185


### Prepare input files for CNV online tool

In [5]:
# CNV data - separated into hg19, hg38, and hg18
# These will be used to input into the CNV prediction tool
genetics_df = pd.read_csv(genetics_only_data)
genetics_df['Human Genome Version'].astype(str)

selected_columns = ['Sample.ID','Sex','CHR','START','STOP','TYPE']
df_38 = genetics_df[genetics_df['Human Genome Version'] == 'Hg38'][selected_columns]
df_19 = genetics_df[genetics_df['Human Genome Version'] == 'Hg19'][selected_columns]
df_18 = genetics_df[genetics_df['Human Genome Version'] == 'Hg18'][selected_columns] # Hg18, ignore

# Save the DataFrame as a TSV file without the index column
df_38.to_csv(hg38_input_data, sep='\t', index=False)
df_19.to_csv(hg19_input_data, sep='\t', index=False)
df_18.to_csv(hg18_input_data, sep='\t', index=False) # Hg18, ignore


### Import files

In [6]:
# ---- Import Data ----
# Original data (CSV)
df = pd.read_csv(original_demog_genetics_data)

# Diagnosis and Cognitive tests Data (CSV)
dia_cogn_df = pd.read_csv(original_dia_cogn_data)

# EEG RS features (CSV)
eeg_rs_features_df = pd.read_csv(original_demog_genetics_data)

# CNV prediction outputs from tool
cnv_hg19_df = pd.read_csv(cnv_prediction_hg19_data)
cnv_hg38_df = pd.read_csv(cnv_prediction_hg38_data)

# Map of # id (used in CNV prediction) and Q1K id
id_map = pd.read_csv(id_map)

# Data manipulations

In [7]:
df.columns = df.columns.str.strip()

### Rename columns

In [8]:
# Column mapping dictionary
column_mapping = {
    'Enter in the box participant\'s EEG code as written here :  [intake_arm1][q1k_relative_idgenerated_1] [intake_arm1][q1k_proband_id_1]': 'ParticipantID',
    'Was EEG attempted?': 'EEG_attempted',
    'EEG site:': 'EEG_site',
    'Birthdate': 'Birthdate',
    'EEG date': 'EEG_date',
    'Age at EEG (years)': 'EEG_age',
    'Sex at birth:': 'Sex_at_birth',
    'Unknown - Specify:': 'diag_unknown_specify',
    'Other - Specify:': 'diag_other_specify',
    'Medication taken the morning of the EEG': 'medication_at_EEG',
    'Resting state with Rio done?': 'RS_Rio_done',
    'Participant\'s code for resting state with Rio :': 'RS_Rio_code',
    'Resting state done?': 'RS_done',
    'Participant\'s code for resting state :': 'RS_code',
    'Tone Oddball done?': 'TO_done',
    'Participant\'s code for TO': 'TO_code',
    'GO done?': 'GO_done',
    'Participant\'s code for GO:': 'GO_code',
    'VEP done?': 'VEP_done',
    'Participant\'s code for VEP:': 'VEP_code',
    'AEP done?': 'AEP_done',
    'Participant\'s code for AEP :   Choose version A or B': 'AEP_code',
    'Randomization file used (A or B)': 'AEP_randomization_file',
    'NSP done?': 'NSP_done',
    'Participant\'s code for NSP:': 'NSP_code',
    'VS done?': 'VS_done',
    'Participant\'s code for VS:': 'VS_code',
    'MMN Oddball done?': 'MMN_done',
    'Participant\'s code for MMN': 'MMN_code',
    'Result aCGH/ LP-WGS': 'Genetic_test_result',
    'Genetic status of the participant:': 'Genetic_status',
    'Affected chromosome:': 'Affected_chromosome',
    'Full proximal boundary (e.g., 2960000):': 'Proximal_boundary',
    'Full distal boundary (e.g., 3020000):': 'Distal_boundary',
    'Please indicate the Human Genome Version used': 'Genome_version',
    'Single gene testing:': 'Single_gene_testing',
    'Fragile X': 'Fragile_X',
    'Exome / Panel testing:': 'Exome_panel_testing',
    'Diagnosis (choice=Control (no genetic or neurodev disorder))': 'diag_control',
    'Diagnosis (choice=Neurodevelopmental disorder)': 'diag_neurodev',
    'Diagnosis (choice=Genetic carrier)': 'diag_genetic_carrier',
    'Diagnosis (choice=Unknown (under investigation, suspected))': 'diag_unknown',
    'Diagnosis (choice=Other (non neurodevelopmental diagnosis))': 'diag_other',
    'Inheritance (choice=De novo)': 'inheritance_denovo',
    'Inheritance (choice=Mothers inherited)': 'inheritance_mothers_inherited',
    'Inheritance (choice=Fathers inherited)': 'inheritance_fathers_inherited',
    'Inheritance (choice=Unknown)': 'inheritance_unknown',
    'Inheritance (choice=Mosaic)': 'inheritance_mosaic'
}

In [9]:
def categorize_family_member_type(id_value):
    """Determine the family member type based on the ID."""
    last_part = id_value.split('_')[-1]
    if last_part == 'P':
        return 'Proband'
    elif last_part.startswith('S') and last_part[1:].isdigit():
        return 'Sibling'
    elif last_part.startswith('F') and last_part[1:].isdigit():
        return 'Father'
    elif last_part.startswith('M') and last_part[1:].isdigit():
        return 'Mother'
    elif last_part.startswith('C') and last_part[1:].isdigit():
        return 'Child'
    elif last_part.startswith('O') and last_part[1:].isdigit():
        return 'Other'
    else:
        return pd.NA

In [10]:
# Keep only first age column and rename it
age_cols = [col for col in df.columns if col == "Age in years"]
df = df.rename(columns={age_cols[0]: "Age at EEG (years)"})

# Rename columns using the mapping dictionary
df = df.rename(columns=column_mapping)

# Add family member type
df['ParticipantID'] = df['ParticipantID'].astype('str')
df['family_member_type'] = df['ParticipantID'].apply(categorize_family_member_type)

In [11]:
# Keep only the columns we want
columns_to_keep = list(column_mapping.values()) + ['family_member_type'] + ['Record ID']
df = df[columns_to_keep]

# Group by Record ID and aggregate
df = df.groupby('Record ID').agg({
    'ParticipantID': lambda x: next((i for i in x if pd.notna(i) and i != 'nan'), pd.NA),
    # Keep first non-null value for all other columns
    **{col: 'first' for col in df.columns if col not in ['Record ID', 'ParticipantID']}
}).reset_index()


# Drop Record ID column
df = df.drop(columns=['Record ID'])

# Convert diagnosis and inheritance columns from Checked/Unchecked to binary 1/0
diag_cols = [col for col in df.columns if col.startswith('diag_')]
inheritance_cols = [col for col in df.columns if col.startswith('inheritance_')]
for col in diag_cols + inheritance_cols:
    df[col] = df[col].map({'Checked': 1, 'Unchecked': 0})


### Merge CNV to original DF

In [12]:
# Merge participantID to the genetic data
cnv_hg19_df = cnv_hg19_df.merge(id_map, on='ID', how = 'left')
cnv_hg38_df = cnv_hg38_df.merge(id_map, on='ID', how = 'left')

# Merge hg19 and hg38 dataframes
cnv_df = pd.concat([cnv_hg19_df, cnv_hg38_df], axis=0)

# Force ParticipantID to be a string
cnv_df['ParticipantID'] = cnv_df['ParticipantID'].astype(str).str.strip()
df['ParticipantID'] = df['ParticipantID'].astype(str).str.strip()
cnv_df.columns = cnv_df.columns.str.strip()
df.columns = df.columns.str.strip()

In [13]:
# Select genetic columns of interest
selected_columns = ['ParticipantID', 'NVIQ_CIupr', 'ORASD_upr', 'SRS_CIupr', 'PdN_CIupr', 'sum_LOEUF_complete']
cnv_selected = cnv_df[selected_columns]

# cnv_selected = cnv_selected.rename(
#     columns={
#         'NVIQ_CIupr': 'Estimated loss of Non-Verbal Intelligence Quotient',
#         'ORASD_upr': 'Estimated odds ratio for autism',
#         'SRS_CIupr': 'Estimated gain of raw score of Social Responsiveness Scale',
#         'PdN_CIupr': 'Estimated probability of being de novo',
#         'sum_LOEUF_complete': 'Sum LOEUF'
#     }
# )

# Merge
df = df.merge(cnv_selected, on='ParticipantID', how='left')

### Merge diagnosis and cognitive tests

In [14]:
# Strip leading and trailing spaces from all column names
dia_cogn_df.columns = dia_cogn_df.columns.str.strip()

# Create column ParticipantID and make sure no leading or trailing spaces
dia_cogn_df['ParticipantID'] = dia_cogn_df['eeg_participant_code'].astype(str).str.strip()
dia_cogn_df.columns = dia_cogn_df.columns.str.strip()

In [15]:
# Merge the two DataFrames on ParticipantID
merged_df = df.merge(dia_cogn_df, on="ParticipantID", how="left")

In [16]:
# Drop duplicated columns
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

In [17]:
# Drop the specified columns
columns_to_drop = ['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance', 'eeg_participant_code']
merged_df = merged_df.drop(columns=columns_to_drop)

In [20]:
# Strip leading and trailing spaces from all string values in the DataFrame
merged_df = merged_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

/var/folders/7j/mcx19g313_vgs3_tv_rpqmrw0000gn/T/ipykernel_44735/4041133145.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged_df = merged_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


### Merge RS EEG features

In [29]:
# Convert CSV to dataframe
rs_eeg_features_df = pd.read_csv(original_eeg_rs_data)

# Strip leading and trailing spaces from all column names
rs_eeg_features_df.columns = rs_eeg_features_df.columns.str.strip()


# Create 2 distinct df for RS Rio vs pure RS
rs_df = rs_eeg_features_df[rs_eeg_features_df['ID'].str.contains('_RS_')].copy()
rsrio_df = rs_eeg_features_df[rs_eeg_features_df['ID'].str.contains('_RSRIO_')].copy()

# Cleanup participant ID
rs_df['ParticipantID'] = rs_df['ID'].str.replace(r'_RS_.*', '', regex=True)
rsrio_df['ParticipantID'] = rsrio_df['ID'].str.replace(r'_RSRIO_.*', '', regex=True)

# Identify EEG features columns
rs_df = rs_df.rename(columns={col: f"EEG_{col}" for col in rs_df.columns if col not in ['ID', 'ParticipantID']})
rsrio_df = rsrio_df.rename(columns={col: f"EEG_{col}" for col in rsrio_df.columns if col not in ['ID', 'ParticipantID']})


In [30]:
# Merge the RS dataframe on ParticipantID
if isRio: rs_data = rsrio_df
else: rs_data = rs_df

merged_df = merged_df.merge(rs_data, on="ParticipantID", how="left")

# Download DB as CSV

In [31]:
final_df = merged_df.copy()

In [32]:
final_df.to_csv(preprocessed_data_path, index=False)
FileLink(preprocessed_data_path)

/Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/Data/Final/GENIAL-DB-preprocessed-RS.csv

# Diving into the data...

## Create a subset for probands only

Remove rows without genetic testing and identify those with normal VS abnormal (or VUS = variant of uncertain significance) genetic status

In [33]:
# Subset with only final_df['family_member_type'] == 'Proband'
proband_df = final_df[final_df['family_member_type'] == 'Proband']
proband_df = proband_df.drop(columns=['family_member_type'])

# Count number of rows in proband_df
print(f"Number of probands: {len(proband_df)}")


Number of probands: 63


In [34]:
# Count rows where diag_neurodev = 1
count_neurodev = (proband_df['diag_neurodev'] == 1).sum()
print(f"Number of probands with neurodevelopmental diagnosis: {count_neurodev}")

Number of probands with neurodevelopmental diagnosis: 59


In [35]:
# Remove rows where Genetic Status is missing
proband_df = proband_df[proband_df['diag_genetic_carrier'].notna()]

# Count number of rows with diag_genetic_carrier = 1
count_genetic_carrier = (proband_df['diag_genetic_carrier'] == 1).sum()
print(f"Number of probands with genetic diagnosis: {count_genetic_carrier}")


Number of probands with genetic diagnosis: 37


In [36]:
# Count number of rows with both neurodev and genetic diagnosis
count_both = (proband_df['diag_neurodev'] == 1) & (proband_df['diag_genetic_carrier'] == 1)
print(f"Number of probands with both neurodevelopmental and genetic diagnosis: {count_both.sum()}")

Number of probands with both neurodevelopmental and genetic diagnosis: 35


## Other family members with diagnostic

In [37]:
# Subset of other family members
other_fam_df = final_df[final_df['family_member_type'] != 'Proband']

# Neurodev count
count_neurodev_diag = (other_fam_df['diag_neurodev'] == 1).sum()
print(f"Number of other family members with neurodevelopmental diagnosis: {count_neurodev_diag}")

# Remove rows where Genetic Status is missing
other_fam_df = other_fam_df[other_fam_df['diag_genetic_carrier'].notna()]

# Genetic abnormality count
count_genetic_diag = (other_fam_df['diag_genetic_carrier'] == 1).sum()

print(f"Number of other family members with genetic diagnosis: {count_genetic_diag}")

Number of other family members with neurodevelopmental diagnosis: 38
Number of other family members with genetic diagnosis: 10
